# State gaming source inventory

This notebook reads the researched inventory and the latest coverage table.
There is **no downloading** here.

Coverage statuses you will see later:

| Status | Meaning |
| --- | --- |
| `ok` | Official revenue (or handle+tax) rows were collected |
| `partial` | Some periods collected; others missing or unpublished |
| `blocked` | Access control, captcha, or no clean online/retail split |
| `not_publicly_available` | Official site does not publish a usable series |
| `combined_only` | Official report mixes products or channels |
| `pdf_only_not_yet_parsed` | Public PDFs exist; parser not promoted yet |
| `legal_not_reporting` | Authorized but no operating reports yet |
| `on_premises_only` / `location_based_mobile` / `annual_only` | Special cases kept out of a simple online comparison |

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config").exists() and (ROOT.parent / "config").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.common import project_root
from variant_gaming.storage import connect, default_db_path

ROOT = project_root()
inventory = pd.read_csv(ROOT / "config" / "state_gaming_source_inventory.csv")
print(len(inventory), "inventory rows")
inventory[["jurisdiction", "state_code", "vertical", "recommended_wave"]].head(12)

42 inventory rows


,jurisdiction,state_code,vertical,recommended_wave
0,Arizona,AZ,online_sports_betting,3
1,Arkansas,AR,online_sports_betting,5
2,Colorado,CO,online_sports_betting,3
3,Connecticut,CT,online_sports_betting,4
4,Connecticut,CT,online_casino,4
5,Delaware,DE,online_sports_betting,2
6,Delaware,DE,online_casino,2
7,District of Columbia,DC,online_sports_betting,2
8,Florida,FL,online_sports_betting,5
9,Illinois,IL,online_sports_betting,1


## Columns

| Column | Meaning |
| --- | --- |
| `jurisdiction` | State or district name |
| `state_code` | Postal code |
| `vertical` | Product (`online_sports_betting` or `online_casino` in the primary table) |
| `official_landing_url` | Regulator page |
| `typical_format` | CSV, Excel, PDF, dashboard |
| `public_granularity` | Operator vs statewide |
| `recommended_wave` | Collection order (1 first) |
| `status_note` | Research caveats |

In [2]:
wave_counts = (
    inventory.groupby("recommended_wave", dropna=False)
    .size()
    .reset_index(name="source_count")
    .sort_values("recommended_wave")
)
wave_counts

,recommended_wave,source_count
0,1,10
1,2,8
2,3,15
3,4,2
4,5,7


## Wave 1 starter sources

In [3]:
inventory.loc[inventory["recommended_wave"] == 1, [
    "jurisdiction", "state_code", "vertical", "official_landing_url", "status_note"
]].sort_values(["state_code", "vertical"])

,jurisdiction,state_code,vertical,official_landing_url,status_note
9,Illinois,IL,online_sports_betting,https://igb.illinois.gov/sports-wagering/sport...,Excellent starter source: official monthly CSV...
10,Indiana,IN,online_sports_betting,https://www.in.gov/igc/publications/monthly-re...,Official monthly revenue workbooks; online det...
17,Maryland,MD,online_sports_betting,https://www.mdgaming.com/maryland-sports-wager...,Excellent starter source: monthly releases lin...
20,Michigan,MI,online_casino,https://www.michigan.gov/mgcb/detroit-casinos/...,Excellent starter source: separate internet-ga...
19,Michigan,MI,online_sports_betting,https://www.michigan.gov/mgcb/detroit-casinos/...,Excellent starter source: annual Excel workboo...
22,Missouri,MO,online_sports_betting,https://www.mgc.dps.mo.gov/SportsWagering/sw_f...,Excellent starter source: official monthly fin...
28,New York,NY,online_sports_betting,https://gaming.ny.gov/revenue-reports,Excellent starter source: official monthly and...
33,Pennsylvania,PA,online_casino,https://gamingcontrolboard.pa.gov/news-and-tra...,Select Interactive Gaming Revenue on the offic...
32,Pennsylvania,PA,online_sports_betting,https://gamingcontrolboard.pa.gov/news-and-tra...,Excellent starter source: fiscal-year Excel re...
36,Tennessee,TN,online_sports_betting,https://www.tn.gov/swac/reports,Excellent starter source: official monthly CSV...


## Coverage already recorded in SQLite (if present)

In [4]:
conn = connect(default_db_path(ROOT))
try:
    coverage = pd.read_sql_query(
        "SELECT state_code, vertical, status, earliest_period, latest_period, normalized_row_count FROM source_coverage ORDER BY state_code, vertical",
        conn,
    )
except Exception:
    coverage = pd.DataFrame()
conn.close()
if coverage.empty:
    print("No source_coverage table yet. Run notebook 20.")
else:
    print(coverage.groupby("status").size())
coverage.head(15) if not coverage.empty else coverage

status
annual_only                       1
blocked                           3
blocked_or_unavailable_export     1
combined_only                     2
legal_not_reporting               1
location_based_mobile             1
not_publicly_available            2
ok                               19
on_premises_only                  1
partial                           4
pdf_only_not_yet_parsed           7
dtype: int64


,state_code,vertical,status,earliest_period,latest_period,normalized_row_count
0,AR,online_sports_betting,combined_only,NaN,NaN,0
1,AZ,online_sports_betting,blocked,NaN,NaN,0
2,CO,online_sports_betting,pdf_only_not_yet_parsed,NaN,NaN,0
3,CT,online_casino,ok,2021-10-01,2026-07-31,223
4,CT,online_sports_betting,ok,2021-10-01,2026-07-31,174
5,DC,online_sports_betting,ok,2024-07-01,2026-07-31,62
6,DE,online_casino,ok,2013-11-01,2026-07-31,612
7,DE,online_sports_betting,blocked,NaN,NaN,0
8,FL,online_sports_betting,not_publicly_available,NaN,NaN,0
9,IA,online_sports_betting,ok,2019-08-01,2026-07-31,1669
